# ModernTCN — OFAT hyperparameter sensitivity, per horizon

One-factor-at-a-time sensitivity for **ModernTCN** at **h = 1, 5 and 22**, run
**separately** for each horizon: every hyperparameter is pinned at *that horizon's*
tuned anchor and one is varied at a time over the Optuna search grid. Each point
trains `ITR` seeds, so every curve carries a mean ± std.

Target and split are the current ones — `Y_t^(h) = ln((1/h) Σ RV_{t+k})` on
`data/EURUSD-RV.csv`, test rows 647 / 643 / 626.

Per horizon you get response curves (MSE + QLIKE), a tornado of signed swing vs
the anchor, a ranked sensitivity bar, and a summary CSV — plus one **cross-horizon**
figure showing which knobs matter at which horizon.

> **Cost.** A full sweep is ~40 configs × `ITR` seeds × 3 horizons. The defaults
> below sweep a focused subset so the notebook finishes in a sitting; widen
> `PARAMS` once you know it works. The sweep is **resumable** — re-running skips
> any `(param, value)` already in the CSV, so you can add parameters later and
> only the new points train.

**Runtime:** `Runtime → Change runtime type → GPU`.

## 1 · Setup

In [ ]:
import os, subprocess, sys

REPO   = "https://github.com/Mr0022/ProjectA.git"
BRANCH = "claude/optimistic-mccarthy-6zszk7"
DIR    = "/content/ProjectA"

if not os.path.isdir(DIR):
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPO,DIR], check=True)
else:
    subprocess.run(["git","-C",DIR,"fetch","origin",BRANCH], check=True)
    subprocess.run(["git","-C",DIR,"checkout",BRANCH], check=True)
    subprocess.run(["git","-C",DIR,"reset","--hard",f"origin/{BRANCH}"], check=True)

os.chdir(DIR); sys.path.insert(0, DIR)
print("HEAD:", subprocess.run(["git","log","--oneline","-1"],
                              capture_output=True, text=True).stdout.strip())

# Guard: the aggregated target must be log(mean RV) = logsumexp(ln_RV) - log(h).
# An older checkout used log(sum RV) and silently inflates MSE (~7x at h=22).
src = open("exp/exp_ModernTCN.py").read()
assert "math.log(h)" in src and "torch.logsumexp" in src, (
    f"stale checkout — delete {DIR} and re-run this cell")
print("target check: OK  (log(mean RV))")

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 2 · Configuration

In [ ]:
HORIZONS = [1, 5, 22]

# Which knobs to sweep. The full list is in sensitivity/ofat_sensitivity.py:ORDER
#   seq_len patch_size patch_stride dim ffn_ratio large_size small_size
#   num_blocks dropout head_dropout learning_rate batch_size revin
# Start focused; widen once a pass completes (the sweep resumes, so nothing reruns).
PARAMS = ["learning_rate", "dropout", "seq_len", "dim"]

ITR    = 3     # seeds per point (paper-grade: 5)
EPOCHS = 20    # max epochs per seed (paper-grade: 40)

# Sanity: print the plan for every horizon without training anything.
for h in HORIZONS:
    print(f"\n========== h = {h} ==========")
    subprocess.run([sys.executable, "sensitivity/ofat_sensitivity.py",
                    "--pred_len", str(h), "--params", *PARAMS,
                    "--itr", str(ITR), "--train_epochs", str(EPOCHS),
                    "--dry_run"], check=True,
                   stdout=subprocess.PIPE, text=True).stdout.splitlines()[:1]
    out = subprocess.run([sys.executable, "sensitivity/ofat_sensitivity.py",
                          "--pred_len", str(h), "--params", *PARAMS,
                          "--itr", str(ITR), "--train_epochs", str(EPOCHS),
                          "--dry_run"], capture_output=True, text=True).stdout
    print("\n".join(l for l in out.splitlines() if l.startswith(("OFAT plan", "   - "))))

## 3 · Run the sweep — one horizon at a time

Each horizon writes its own CSV (`sensitivity/ofat_moderntcn_h<H>.csv`), so the
three studies never mix. Progress streams below; interrupt any time and re-run —
completed points are skipped.

In [ ]:
import re, time

def stream(cmd, keep=r"^(\[OK\]|\[WARN\]|OFAT plan|##########|\[OFAT\])"):
    pat, buf = re.compile(keep), []
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1)
    for line in pr.stdout:
        buf.append(line)
        if pat.search(line):
            print(line.rstrip())
    pr.wait()
    if pr.returncode != 0:
        print("".join(buf[-25:]))
        raise RuntimeError(f"exit {pr.returncode}")
    return "".join(buf)

for h in HORIZONS:
    print("\n" + "=" * 72)
    print(f"OFAT sweep — ModernTCN, h = {h}")
    print("=" * 72)
    t0 = time.time()
    stream([sys.executable, "sensitivity/ofat_sensitivity.py",
            "--pred_len", str(h), "--params", *PARAMS,
            "--itr", str(ITR), "--train_epochs", str(EPOCHS)])
    print(f"--> h={h} done in {(time.time()-t0)/60:.1f} min")

## 4 · Figures — per horizon, then cross-horizon

Response curves are one series per panel (mean line, ±1 std band, per-seed dots,
◆ = tuned anchor). The cross-horizon bar uses one validated colour per horizon.

In [ ]:
for h in HORIZONS:
    print(f"\n--- h = {h} ---")
    stream([sys.executable, "sensitivity/ofat_plots.py", "--pred_len", str(h)],
           keep=r"^(wrote|Figures)")

stream([sys.executable, "sensitivity/ofat_plots.py",
        "--compare", *[str(h) for h in HORIZONS]], keep=r"^(wrote|Cross)")

In [ ]:
from IPython.display import Image, Markdown, display

for h in HORIZONS:
    display(Markdown(f"### h = {h}"))
    for name in ("ofat_response_mse", "ofat_response_qlike",
                 "ofat_tornado_mse", "ofat_sensitivity_bar"):
        p = f"sensitivity/figures/h{h}/{name}.png"
        if os.path.exists(p):
            display(Image(filename=p))

display(Markdown("### All horizons"))
display(Image(filename="sensitivity/figures/ofat_cross_horizon_mse.png"))

## 5 · Summary table

`ofat_summary.csv` holds every point's mean ± std. Below, the knobs ranked by how
far they move test MSE around each horizon's anchor — the numbers behind the
cross-horizon figure, and a text fallback for the contrast WARN on the aqua bar.

In [ ]:
import numpy as np, pandas as pd

rows = []
for h in HORIZONS:
    s = pd.read_csv(f"sensitivity/figures/h{h}/ofat_summary.csv")
    s = s[s.metric == "mse"]
    a = float(s.loc[s.is_anchor.astype(bool), "mean"].mean())
    for p, g in s.groupby("param"):
        if len(g) < 2:
            continue
        rows.append(dict(horizon=h, param=p,
                         best=g["mean"].min(), worst=g["mean"].max(),
                         anchor=a, swing_pct=(g["mean"].max()-g["mean"].min())/a*100))
tbl = pd.DataFrame(rows)
if not tbl.empty:
    wide = (tbl.pivot(index="param", columns="horizon", values="swing_pct")
              .rename(columns=lambda h: f"h={h}"))
    wide["mean"] = wide.mean(axis=1)
    wide = wide.sort_values("mean", ascending=False)
    print("MSE swing across the grid, as % of that horizon's anchor MSE")
    print("(higher = the model is more sensitive to that knob)\n")
    display(wide.round(1))
    tbl.to_csv("sensitivity/ofat_sensitivity_by_horizon.csv", index=False)
    print("\nsaved sensitivity/ofat_sensitivity_by_horizon.csv")

In [ ]:
# Bundle every figure + CSV for download
import shutil
shutil.make_archive("/content/ofat_sensitivity", "zip", "sensitivity/figures")
try:
    from google.colab import files
    files.download("/content/ofat_sensitivity.zip")
except Exception as e:
    print(f"(figures in {os.getcwd()}/sensitivity/figures; "
          f"auto-download unavailable: {type(e).__name__})")

---

### What changed versus the code you had

The old `sensitivity/` study was **EventTCN at h = 1 only**, hardcoded. It now:

- takes `--pred_len {1,5,22}`, each anchored at its **own** tuned config from
  `tuningresults/ModernTCN{1,5,22}`, writing a separate CSV per horizon;
- defaults to **plain ModernTCN**; `--use_events` restores the EventTCN study and
  re-adds `event_dim` to the grid;
- reads `EURUSD-RV.csv` / `--target RV` / `--aggregate_mean`, matching the current
  data, target and split;
- writes figures to `sensitivity/figures/h<H>/` and adds a cross-horizon figure.

**A palette fix:** the anchor marker used to be red `#e34948` on an orange `#eb6834`
QLIKE curve — normal-vision ΔE 7.1, below the 15 floor, i.e. hard to tell apart even
with full colour vision. The anchor is an annotation, not a series, so it is now an
ink diamond with a direct "tuned" label, and the three horizon colours are
categorical slots 1–3, validated all-pairs on both light and dark surfaces.

### Caveats

- OFAT is **local**: it measures sensitivity around each anchor, not a global
  variance decomposition. For the global view use the Optuna `param_importances`
  artifacts in `tuningresults/`.
- The anchors were tuned on the **previous** dataset and target, so they are not
  guaranteed optimal here. The sweep is still a valid local study; re-run `tune.py`
  if you want anchors that are optimal for the current setup.
- `patch_stride` is clamped to `min(patch_stride, patch_size)`, exactly as
  `tune.py` does, so the `patch_size` sweep never yields a stride wider than the patch.